<a href="https://colab.research.google.com/github/03sarath/vertex-ai-mlops-kfp2/blob/main/Vertex_AI_kfp2_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Copyright & License (click to expand)



In [ ]:
# @title Copyright & License (click to expand)
# Copyright 2023 Psitron Technologies Pvt Ltd
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

## Overview

This tutorial uses the Vertex AI Pipelines with KFP 2.x.


### Objective: **🍷** **Predicting wine quality**

In this tutorial, you learn to use `Vertex AI Pipelines` and KFP 2.x version of `Google Cloud Pipeline Components` to train and deploy an RandomForestClassifier model.


This tutorial uses the following Google Cloud ML services:

- `Vertex AI Pipelines`
- `Google Cloud Pipeline Components`

The steps performed include:

- Create a KFP pipeline:
    - Data preprocessing.
    - Train an RandomForestClassifier `Model`.
    - Evaluation an `Model`.
    - Create an `Endpoint` resource.
    - Deploys the `Model` resource to the `Endpoint` resource.
- Compile the KFP pipeline.
- Execute the KFP pipeline using `Vertex AI Pipelines`

The components are [documented here](https://google-cloud-pipeline-components.readthedocs.io/en/google-cloud-pipeline-components-1.0.0/google_cloud_pipeline_components.aiplatform.html).

### Install Vertex AI SDK for Python and other required packages

In [1]:
! pip3 install --no-cache-dir --upgrade "kfp>2" \
                                        google-cloud-aiplatform

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.4/63.4 kB 19.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.5/397.5 kB 198.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.2/323.2 kB 37.6 MB/s eta 0:00:00
  Created wheel for kfp-server-api: filename=kfp_server_api-2.15.1-py3-none-any.whl size=113922 sha256=badea37bb26637464ee5c6756b11aab46049f8795f767ce4cf7550797f8bb324
  Stored in directory: /tmp/pip-ephem-wheel-cache-aa1wb3f7/wheels/0a/d5/c2/2a529d1ab26cd9ad4cb54fcccb2de4de793d4cf6cf8a732dd3
Successfully built kfp-server-api
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.

Check the KFP SDK version.

In [2]:

! python3 -c "import kfp; print('KFP SDK version: {}'.format(kfp.__version__))"
! pip3 freeze | grep aiplatform

KFP SDK version: 2.15.1
google-cloud-aiplatform==1.128.0


### Restart runtime (Colab only)

To use the newly installed packages, you must restart the runtime on Google Colab.

In [59]:
import sys

if "google.colab" in sys.modules:

    import IPython

    app = IPython.Application.instance()
    app.kernel.do_shutdown(True)

### Authenticate your notebook environment (Colab only)

Authenticate your environment on Google Colab.

In [1]:

import sys

if "google.colab" in sys.modules:

    from google.colab import auth

    auth.authenticate_user()

### Set Google Cloud project information

Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [2]:

PROJECT_ID = "test-vertex-479702"  # @param {type:"string"}
LOCATION = "us-central1"
BQ_LOCATION = LOCATION.split("-")[0].upper()

### Create a Cloud Storage bucket

Create a storage bucket to store intermediate artifacts such as datasets.

In [3]:
BUCKET_URI = f"gs://your-bucket-name-{PROJECT_ID}-unique"  # @param {type:"string"}


In [4]:
# Create bucket
PIPELINE_ROOT = f"{BUCKET_URI}/pipeline_root_wine/"
PIPELINE_ROOT

'gs://your-bucket-name-test-vertex-479702-unique/pipeline_root_wine/'

**Only if your bucket doesn't already exist**: Run the following cell to create your Cloud Storage bucket.

In [5]:
! gsutil mb -l $LOCATION -p $PROJECT_ID $BUCKET_URI

Creating gs://your-bucket-name-test-vertex-479702-unique/...
ServiceException: 409 A Cloud Storage bucket named 'your-bucket-name-test-vertex-479702-unique' already exists. Try another name. Bucket names must be globally unique across all Google Cloud projects, including those outside of your organization.


#### Service Account

You use a service account to create Vertex AI Pipeline jobs.

In [6]:
SERVICE_ACCOUNT = "[your-service-account]"  # @param {type:"string"}

In [7]:
import sys

IS_COLAB = "google.colab" in sys.modules
if (
    SERVICE_ACCOUNT == ""
    or SERVICE_ACCOUNT is None
    or SERVICE_ACCOUNT == "[your-service-account]"
):
    # Get your service account from gcloud
    if not IS_COLAB:
        shell_output = !gcloud auth list 2>/dev/null
        SERVICE_ACCOUNT = shell_output[2].replace("*", "").strip()

    else:  # IS_COLAB:
        shell_output = ! gcloud projects describe  $PROJECT_ID
        project_number = shell_output[-1].split(":")[1].strip().replace("'", "")
        SERVICE_ACCOUNT = f"{project_number}-compute@developer.gserviceaccount.com"

    print("Service Account:", SERVICE_ACCOUNT)

Service Account: 726895148784-compute@developer.gserviceaccount.com


#### Set service account access for Vertex AI Pipelines

Run the following commands to grant your service account access to read and write pipeline artifacts in the bucket that you created in the previous step. You only need to run this step once per service account.

In [8]:
! gsutil iam ch serviceAccount:{SERVICE_ACCOUNT}:roles/storage.objectCreator $BUCKET_URI

! gsutil iam ch serviceAccount:{SERVICE_ACCOUNT}:roles/storage.objectViewer $BUCKET_URI

No changes made to gs://your-bucket-name-test-vertex-479702-unique/
No changes made to gs://your-bucket-name-test-vertex-479702-unique/


### Import libraries and define constants

In [9]:
import google.cloud.aiplatform as aiplatform
import kfp
from kfp import compiler, dsl
from kfp.dsl import Artifact, Dataset, Input, Metrics, Model, Output, component, ClassificationMetrics
from typing import NamedTuple
from google.cloud.aiplatform import pipeline_jobs


## Initialize Vertex AI SDK for Python

Initialize the Vertex AI SDK for Python for your project and corresponding bucket.

In [10]:
aiplatform.init(project=PROJECT_ID, staging_bucket=BUCKET_URI)

### Define Data preparation component


In [11]:
@component(
    packages_to_install=["numpy==2.1.2", "pandas==2.2.2", "pyarrow", "scikit-learn==1.4.2"],
)

def get_wine_data(
    url: str,
    dataset_train: Output[Dataset],
    dataset_test: Output[Dataset]
):
    import pandas as pd
    import numpy as np
    from sklearn.model_selection import train_test_split as tts

    df_wine = pd.read_csv(url, delimiter=";")
    df_wine['best_quality'] = [ 1 if x>=7 else 0 for x in df_wine.quality]
    df_wine['target'] = df_wine.best_quality
    df_wine = df_wine.drop(['quality', 'total sulfur dioxide', 'best_quality'], axis=1)


    train, test = tts(df_wine, test_size=0.3)
    train.to_csv(dataset_train.path + ".csv" , index=False, encoding='utf-8-sig')
    test.to_csv(dataset_test.path + ".csv" , index=False, encoding='utf-8-sig')


/tmp/ipython-input-4004937912.py:1: FutureWarning: The default base_image used by the @dsl.component decorator will switch from 'python:3.11' to 'python:3.12' on Oct 1, 2027. To ensure your existing components work with versions of the KFP SDK released after that date, you should provide an explicit base_image argument and ensure your component works as intended on Python 3.12.
  @component(


### Define RandomForestClassifier Model training component


In [12]:
@component(
    packages_to_install = [
        "numpy==2.1.2",
        "pandas==2.2.2",
        "scikit-learn==1.4.2"
    ]
)
def train_winequality(
    dataset:  Input[Dataset],
    model: Output[Model],
):

    from sklearn.ensemble import RandomForestClassifier
    import pandas as pd
    import pickle

    data = pd.read_csv(dataset.path+".csv")
    model_rf = RandomForestClassifier(n_estimators=10)
    model_rf.fit(
        data.drop(columns=["target"]),
        data.target,
    )
    model.metadata["framework"] = "RF"
    file_name = model.path + f".pkl"
    with open(file_name, 'wb') as file:
        pickle.dump(model_rf, file)

/tmp/ipython-input-201807934.py:1: FutureWarning: The default base_image used by the @dsl.component decorator will switch from 'python:3.11' to 'python:3.12' on Oct 1, 2027. To ensure your existing components work with versions of the KFP SDK released after that date, you should provide an explicit base_image argument and ensure your component works as intended on Python 3.12.
  @component(


### Define Model evaluation component


In [13]:
@component(
    packages_to_install = [
        "numpy==2.1.2",
        "pandas==2.2.2",
        "scikit-learn==1.4.2"
    ]
)
def winequality_evaluation(
    test_set:  Input[Dataset],
    rf_winequality_model: Input[Model],
    thresholds_dict_str: str,
    metrics: Output[ClassificationMetrics],
    kpi: Output[Metrics]
) -> NamedTuple("output", [("deploy", str)]):

    import pandas as pd
    import pickle
    import json
    import math
    from sklearn.metrics import roc_curve, confusion_matrix, accuracy_score

    def threshold_check(val1, val2):
        return "true" if val1 >= val2 else "false"

    # Load data
    data = pd.read_csv(test_set.path + ".csv")

    # Load model
    with open(rf_winequality_model.path + ".pkl", 'rb') as file:
        model = pickle.load(file)

    # Predictions
    y_test = data.drop(columns=["target"])
    y_target = data["target"].astype(int)  # ensure native ints
    y_pred = model.predict(y_test)

    # ROC curve
    y_scores = model.predict_proba(y_test)[:, 1]
    fpr, tpr, thresholds = roc_curve(y_true=y_target, y_score=y_scores, pos_label=1)

    # Sanitize Infinity values: replace any ±inf with the max finite threshold (or 1.0 if none)
    finite_thresholds = [t for t in thresholds if not math.isinf(t) and not math.isnan(t)]
    if finite_thresholds:
        max_finite = max(finite_thresholds)
    else:
        max_finite = 1.0
    sanitized_thresholds = [float(t) if (not math.isinf(t) and not math.isnan(t)) else float(max_finite) for t in thresholds]

    # Log ROC curve with lists of native floats
    metrics.log_roc_curve(
        fpr=fpr.tolist(),
        tpr=tpr.tolist(),
        thresholds=[float(x) for x in sanitized_thresholds]
    )

    # Confusion matrix (native Python ints)
    cm = confusion_matrix(y_target.tolist(), y_pred.tolist()).tolist()
    metrics.log_confusion_matrix(["False", "True"], cm)

    # Accuracy
    accuracy = float(accuracy_score(y_target, y_pred))
    thresholds_dict = json.loads(thresholds_dict_str)

    # Store accuracy as model metadata (native float)
    rf_winequality_model.metadata["accuracy"] = float(accuracy)
    kpi.log_metric("accuracy", float(accuracy))

    deploy = threshold_check(float(accuracy), float(thresholds_dict.get("roc", 0.0)))
    return (deploy,)


/tmp/ipython-input-230287119.py:1: FutureWarning: The default base_image used by the @dsl.component decorator will switch from 'python:3.11' to 'python:3.12' on Oct 1, 2027. To ensure your existing components work with versions of the KFP SDK released after that date, you should provide an explicit base_image argument and ensure your component works as intended on Python 3.12.
  @component(


### Define deploying the model component

Finally, you define a component to deploy the model.

In [14]:
@component(
    packages_to_install=["google-cloud-aiplatform", "scikit-learn==1.4.2",  "kfp", "numpy==2.1.2",],
    output_component_file="model_winequality_coponent.yml"
)
def deploy_winequality(
    model: Input[Model],
    project: str,
    region: str,
    serving_container_image_uri : str,
    vertex_endpoint: Output[Artifact],
    vertex_model: Output[Model]
):
    from google.cloud import aiplatform
    aiplatform.init(project=project, location=region)

    DISPLAY_NAME  = "winequality"
    MODEL_NAME = "winequality-rf"
    ENDPOINT_NAME = "winequality_endpoint"

    def create_endpoint():
        endpoints = aiplatform.Endpoint.list(
        filter='display_name="{}"'.format(ENDPOINT_NAME),
        order_by='create_time desc',
        project=project,
        location=region,
        )
        if len(endpoints) > 0:
            endpoint = endpoints[0]  # most recently created
        else:
            endpoint = aiplatform.Endpoint.create(
            display_name=ENDPOINT_NAME, project=project, location=region
        )
    endpoint = create_endpoint()


    #Import a model programmatically
    model_upload = aiplatform.Model.upload(
        display_name = DISPLAY_NAME,
        artifact_uri = model.uri.replace("model", ""),
        serving_container_image_uri =  serving_container_image_uri,
        serving_container_health_route=f"/v1/models/{MODEL_NAME}",
        serving_container_predict_route=f"/v1/models/{MODEL_NAME}:predict",
        serving_container_environment_variables={
        "MODEL_NAME": MODEL_NAME,
    },
    )
    model_deploy = model_upload.deploy(
        machine_type="n1-standard-4",
        endpoint=endpoint,
        traffic_split={"0": 100},
        deployed_model_display_name=DISPLAY_NAME,
    )

    # Save data to the output params
    vertex_model.uri = model_deploy.resource_name

/tmp/ipython-input-3266322023.py:1: DeprecationWarning: output_component_file parameter is deprecated and will eventually be removed. Please use `Compiler().compile()` to compile a component instead.
  @component(
/tmp/ipython-input-3266322023.py:1: FutureWarning: The default base_image used by the @dsl.component decorator will switch from 'python:3.11' to 'python:3.12' on Oct 1, 2027. To ensure your existing components work with versions of the KFP SDK released after that date, you should provide an explicit base_image argument and ensure your component works as intended on Python 3.12.
  @component(


In [15]:
from datetime import datetime

TIMESTAMP =datetime.now().strftime("%Y%m%d%H%M%S")
DISPLAY_NAME = 'pipeline-winequality-job{}'.format(TIMESTAMP)

## Construct the RandomForestClassifier training pipeline

Now you define the pipeline, with the following steps:

- Data preparation.
- Train the model.
- Evaluate the model.
- Deploy the model.

In [16]:
@dsl.pipeline(
    # Default pipeline root. You can override it when submitting the pipeline.
    pipeline_root=PIPELINE_ROOT,
    # A name for the pipeline. Use to determine the pipeline Context.
    name="pipeline-winequality",

)
def pipeline(
    url: str = "https://psitron.s3.ap-southeast-1.amazonaws.com/vertexai/winequality-white.csv",
    project: str = PROJECT_ID,
    region: str = LOCATION,
    display_name: str = DISPLAY_NAME,
    api_endpoint: str = LOCATION+"-aiplatform.googleapis.com",
    thresholds_dict_str: str = '{"roc":0.8}',
    serving_container_image_uri: str = "europe-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.0-24:latest"
    ):

    data_op = get_wine_data(url=url)
    train_model_op = train_winequality(dataset=data_op.outputs["dataset_train"])
    model_evaluation_op = winequality_evaluation(
        test_set=data_op.outputs["dataset_test"],
        rf_winequality_model=train_model_op.outputs["model"],
        thresholds_dict_str = thresholds_dict_str, # I deploy the model anly if the model performance is above the threshold
    )

    with dsl.Condition(
        model_evaluation_op.outputs["deploy"]=="true",
        name="deploy-winequality",
    ):

        deploy_model_op = deploy_winequality(
        model=train_model_op.outputs['model'],
        project=project,
        region=region,
        serving_container_image_uri = serving_container_image_uri,
        )


/tmp/ipython-input-633038914.py:26: DeprecationWarning: dsl.Condition is deprecated. Please use dsl.If instead.
  with dsl.Condition(


### Compile the pipeline

Next, you compile the pipeline.

In [17]:
compiler.Compiler().compile(pipeline_func=pipeline,
        package_path='ml_winequality.json')

### Run the pipeline using Vertex AI Pipelines

Now, run the compiled pipeline.

In [18]:
start_pipeline = pipeline_jobs.PipelineJob(
    display_name="winequality-pipeline",
    template_path="ml_winequality.json",
    enable_caching=False,
    location=LOCATION,
)

In [19]:
start_pipeline.run()


RuntimeError: Job failed with:
code: 9
message: " The DAG failed because some tasks failed. The failed tasks are: [winequality-evaluation].; Job (project_id = test-vertex-479702, job_id = 617278609656119296) is failed due to the above error.; Failed to handle the job: {project_number = 726895148784, job_id = 617278609656119296}"


In [ ]:
import google.cloud.aiplatform as aiplatform
import kfp
from kfp import compiler, dsl
from kfp.dsl import Artifact, Dataset, Input, Metrics, Model, Output, component, ClassificationMetrics
from typing import NamedTuple
from google.cloud.aiplatform import pipeline_jobs
from datetime import datetime

# DEFINE THESE VARIABLES AT THE TOP
PROJECT_ID = "test-vertex-479702"  # Your project ID
LOCATION = "us-central1"  # Your region
PIPELINE_ROOT = f"gs://{PROJECT_ID}-pipeline-root"  # Your GCS bucket

@component(
    packages_to_install=["numpy==1.24.3", "pandas==2.2.2", "pyarrow", "scikit-learn==1.4.2"],
)
def get_wine_data(
    url: str,
    dataset_train: Output[Dataset],
    dataset_test: Output[Dataset]
):
    import pandas as pd
    import numpy as np
    from sklearn.model_selection import train_test_split as tts

    df_wine = pd.read_csv(url, delimiter=";")
    df_wine['best_quality'] = [1 if x >= 7 else 0 for x in df_wine.quality]
    df_wine['target'] = df_wine.best_quality
    df_wine = df_wine.drop(['quality', 'total sulfur dioxide', 'best_quality'], axis=1)

    train, test = tts(df_wine, test_size=0.3, random_state=42)

    train.to_csv(dataset_train.path + ".csv", index=False, encoding='utf-8-sig')
    test.to_csv(dataset_test.path + ".csv", index=False, encoding='utf-8-sig')


@component(
    packages_to_install=["numpy==1.24.3", "pandas==2.2.2", "scikit-learn==1.4.2"]
)
def train_winequality(
    dataset: Input[Dataset],
    model: Output[Model],
):
    from sklearn.ensemble import RandomForestClassifier
    import pandas as pd
    import pickle

    data = pd.read_csv(dataset.path + ".csv")
    model_rf = RandomForestClassifier(n_estimators=10, random_state=42)
    model_rf.fit(
        data.drop(columns=["target"]),
        data.target,
    )

    model.metadata["framework"] = "RF"
    file_name = model.path + f".pkl"
    with open(file_name, 'wb') as file:
        pickle.dump(model_rf, file)


@component(
    packages_to_install=["numpy==1.24.3", "pandas==2.2.2", "scikit-learn==1.4.2"]
)
def winequality_evaluation(
    test_set: Input[Dataset],
    rf_winequality_model: Input[Model],
    thresholds_dict_str: str,
    metrics: Output[ClassificationMetrics],
    kpi: Output[Metrics]
) -> NamedTuple("output", [("deploy", str)]):
    import pandas as pd
    import pickle
    import json
    import numpy as np
    from sklearn.metrics import roc_curve, confusion_matrix, accuracy_score

    def threshold_check(val1, val2):
        return "true" if val1 >= val2 else "false"

    print("Loading test data...")
    # Load data
    data = pd.read_csv(test_set.path + ".csv")
    print(f"Test data shape: {data.shape}")
    print(f"Columns: {data.columns.tolist()}")

    print("Loading model...")
    # Load model
    with open(rf_winequality_model.path + ".pkl", 'rb') as file:
        model = pickle.load(file)

    # Predictions
    print("Making predictions...")
    y_test = data.drop(columns=["target"])
    y_target = data["target"].astype(int)  # Ensure integer type
    y_pred = model.predict(y_test)

    print("Calculating ROC curve...")
    # ROC curve
    y_scores = model.predict_proba(y_test)[:, 1]
    fpr, tpr, thresholds = roc_curve(
        y_true=y_target.values,
        y_score=y_scores,
        pos_label=1  # Use integer 1 instead of True
    )

    # Sanitize values: remove inf/nan, convert to native Python types
    fpr_clean = [float(x) for x in fpr if not (np.isinf(x) or np.isnan(x))]
    tpr_clean = [float(x) for x in tpr if not (np.isinf(x) or np.isnan(x))]
    thresholds_clean = [
        float(t) if not (np.isinf(t) or np.isnan(t)) else 0.0
        for t in thresholds
    ]

    # Ensure all three lists have the same length
    min_len = min(len(fpr_clean), len(tpr_clean), len(thresholds_clean))
    fpr_clean = fpr_clean[:min_len]
    tpr_clean = tpr_clean[:min_len]
    thresholds_clean = thresholds_clean[:min_len]

    print(f"ROC curve points: {len(fpr_clean)}")

    try:
        metrics.log_roc_curve(
            fpr=fpr_clean,
            tpr=tpr_clean,
            thresholds=thresholds_clean
        )
        print("ROC curve logged successfully")
    except Exception as e:
        print(f"Warning: Could not log ROC curve: {str(e)}")

    # Confusion matrix
    print("Calculating confusion matrix...")
    cm = confusion_matrix(y_target, y_pred)
    cm_list = [[int(x) for x in row] for row in cm]  # Convert to native Python ints

    try:
        metrics.log_confusion_matrix(
            ["False", "True"],
            cm_list
        )
        print("Confusion matrix logged successfully")
    except Exception as e:
        print(f"Warning: Could not log confusion matrix: {str(e)}")

    # Accuracy
    print("Calculating accuracy...")
    accuracy = float(accuracy_score(y_target, y_pred))
    print(f"Accuracy: {accuracy:.4f}")

    thresholds_dict = json.loads(thresholds_dict_str)
    threshold_value = float(thresholds_dict["roc"])

    # Store accuracy as model metadata
    rf_winequality_model.metadata["accuracy"] = accuracy
    kpi.log_metric("accuracy", accuracy)

    deploy = threshold_check(accuracy, threshold_value)
    print(f"Deploy decision: {deploy} (accuracy {accuracy:.4f} vs threshold {threshold_value})")

    return (deploy,)


@component(
    packages_to_install=["google-cloud-aiplatform", "scikit-learn==1.4.2", "kfp", "numpy==1.24.3"],
    output_component_file="model_winequality_component.yml"
)
def deploy_winequality(
    model: Input[Model],
    project: str,
    region: str,
    serving_container_image_uri: str,
    vertex_endpoint: Output[Artifact],
    vertex_model: Output[Model]
):
    from google.cloud import aiplatform
    import os

    aiplatform.init(project=project, location=region)

    DISPLAY_NAME = "winequality"
    MODEL_NAME = "winequality-rf"
    ENDPOINT_NAME = "winequality_endpoint"

    def create_endpoint():
        endpoints = aiplatform.Endpoint.list(
            filter='display_name="{}"'.format(ENDPOINT_NAME),
            order_by='create_time desc',
            project=project,
            location=region,
        )
        if len(endpoints) > 0:
            endpoint = endpoints[0]  # most recently created
        else:
            endpoint = aiplatform.Endpoint.create(
                display_name=ENDPOINT_NAME,
                project=project,
                location=region
            )
        return endpoint

    endpoint = create_endpoint()

    # Fix artifact URI - remove the filename, keep only the directory
    artifact_uri = model.uri.rsplit('/', 1)[0] + '/'

    # Import a model programmatically
    model_upload = aiplatform.Model.upload(
        display_name=DISPLAY_NAME,
        artifact_uri=artifact_uri,
        serving_container_image_uri=serving_container_image_uri,
        serving_container_health_route=f"/v1/models/{MODEL_NAME}",
        serving_container_predict_route=f"/v1/models/{MODEL_NAME}:predict",
        serving_container_environment_variables={
            "MODEL_NAME": MODEL_NAME,
        },
    )

    model_deploy = model_upload.deploy(
        machine_type="n1-standard-4",
        endpoint=endpoint,
        traffic_split={"0": 100},
        deployed_model_display_name=DISPLAY_NAME,
    )

    # Save data to the output params
    vertex_model.uri = model_deploy.resource_name


@dsl.pipeline(
    pipeline_root=PIPELINE_ROOT,
    name="pipeline-winequality",
)
def pipeline(
    url: str = "https://psitron.s3.ap-southeast-1.amazonaws.com/vertexai/winequality-white.csv",
    project: str = PROJECT_ID,
    region: str = LOCATION,
    thresholds_dict_str: str = '{"roc":0.8}',
    serving_container_image_uri: str = "us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-0:latest"
):
    data_op = get_wine_data(url=url)

    train_model_op = train_winequality(dataset=data_op.outputs["dataset_train"])

    model_evaluation_op = winequality_evaluation(
        test_set=data_op.outputs["dataset_test"],
        rf_winequality_model=train_model_op.outputs["model"],
        thresholds_dict_str=thresholds_dict_str,
    )

    # Deploy the model only if the model performance is above the threshold
    with dsl.Condition(
        model_evaluation_op.outputs["deploy"] == "true",
        name="deploy-winequality",
    ):
        deploy_model_op = deploy_winequality(
            model=train_model_op.outputs['model'],
            project=project,
            region=region,
            serving_container_image_uri=serving_container_image_uri,
        )


# Compile the pipeline
compiler.Compiler().compile(
    pipeline_func=pipeline,
    package_path='ml_winequality.json'
)

# Create timestamp for display name
TIMESTAMP = datetime.now().strftime("%Y%m%d%H%M%S")
DISPLAY_NAME = f'pipeline-winequality-job{TIMESTAMP}'

# Run the pipeline
start_pipeline = pipeline_jobs.PipelineJob(
    display_name=DISPLAY_NAME,
    template_path="ml_winequality.json",
    enable_caching=False,
    location=LOCATION,
    project=PROJECT_ID,
)

start_pipeline.run()

/tmp/ipython-input-1086675539.py:14: FutureWarning: The default base_image used by the @dsl.component decorator will switch from 'python:3.11' to 'python:3.12' on Oct 1, 2027. To ensure your existing components work with versions of the KFP SDK released after that date, you should provide an explicit base_image argument and ensure your component works as intended on Python 3.12.
  @component(
/tmp/ipython-input-1086675539.py:37: FutureWarning: The default base_image used by the @dsl.component decorator will switch from 'python:3.11' to 'python:3.12' on Oct 1, 2027. To ensure your existing components work with versions of the KFP SDK released after that date, you should provide an explicit base_image argument and ensure your component works as intended on Python 3.12.
  @component(
/tmp/ipython-input-1086675539.py:61: FutureWarning: The default base_image used by the @dsl.component decorator will switch from 'python:3.11' to 'python:3.12' on Oct 1, 2027. To ensure your existing componen

# ⏰ **Schedule** your pipelines to run recurrently

In [ ]:
#Schedule your pipelines to run recurrently

from google.cloud import aiplatform

pipeline_job = aiplatform.PipelineJob(
  template_path="ml_winequality.json",
  pipeline_root=PIPELINE_ROOT,
  display_name="ml_winequality",
)

pipeline_job_schedule = aiplatform.PipelineJobSchedule(
  pipeline_job=pipeline_job,
  display_name="ml_winequality"
)

pipeline_job_schedule.create(
  #cron="0 0 * * *",
  cron="TZ=Asia/Calcutta 0 0 * * *"
  max_concurrent_run_count=1,
  max_run_count=10,
)

#⏰List your all scheduled pipelines


In [ ]:
#List your all scheduled pipelines
from google.cloud import aiplatform

aiplatform.PipelineJobSchedule.list(
  filter='display_name="ml_winequality"',
  order_by='create_time desc'
)

#Invoke **ML model deployed in vertex ai endpoint**

In [ ]:
import subprocess
from google.cloud import aiplatform

ENDPOINT_NAME = "winequality_endpoint"

#Bad Qty Wine Data 👎
instance = [[7.2, 0.23, 0.32, 8.5, 0.058, 47, 132, 0.993, 3.12, 0.46]]

#Good Qty Wine Data 👍
#instance = [[6.7,0.23,0.31,2.1,0.046,30,0.9926,3.33,0.64,10.7]]


def endpoint_predict(project: str, location: str, instances: list, endpoint: str):
    aiplatform.init(project=project, location=location)
    endpoint = aiplatform.Endpoint(endpoint)
    prediction = endpoint.predict(instances=instances)
    return prediction

#prediction = endpoint_predict('<project_id>', 'us-central1', instance, '<endpoint_id>')
prediction = endpoint_predict('test-verstexai', 'us-central1', instance, '5036881458539528192')
print("Prediction from Deployed model_id:", prediction[1])
print("Prediction from Deployed model version_id:", prediction[3])
print("Predicted wine quality 🍷👍👎:", prediction[0])